# Plagiarism Detection
In this assignment, we will sequentially practice the steps to build a plagiarism detection application using a pre-trained Word2Vec model.
Data: https://s3.amazonaws.com/video.udacity-data.com/topher/2019/January/5c4147f9_data/data.zip
This exercise requires knowledge of Python programming with the following libraries:
 * `gensim` (to load the Word2Vec model)
 * `numpy` (to compute similarity)
Additionally, we will use a pre-trained Word2Vec model `Google's pre-trained word2vec model`
Steps to Solve This Exercise
1. Exploring the Dataset  
2. Build a class for computing document similarity (`DocSim` class)
3. Create an instance of the above class (Load the pre-trained word embedding model, Create a list of stopwords and create an instance of the `DocSim` class)
4. Plagiarism Detection and Evaluation

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import os, shutil
import requests
import zipfile
import datetime
print(f"This notebook was last run in: {datetime.datetime.now() : %Y %m %d %H %M %S}" )

This notebook was last run in:  2025 09 30 23 00 45


## Load the dataset

In [ ]:
! cd data/ & wget -O 'local_filename.zip' 'https://s3.amazonaws.com/video.udacity-data.com/topher/2019/January/5c4147f9_data/data.zip'

In [ ]:
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

zip_path = os.path.join(data_dir, "local_filename.zip")
print(zip_path)

def unzip(zip_path, data_dir, delete=True):
    if zip_path.endswith(".zip"):
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        print("Extracted zip to", data_dir)
    else:
        print("This file format is not accepted.")
        return
    
    if delete:
        os.remove(zip_path)
        print("Removed!")
        return
    return

In [ ]:
unzip(zip_path, data_dir)

### Load data_infor file

In [8]:
data_path = os.path.join(data_dir, "data")
file_info_path = os.path.join(data_path, "file_information.csv")
df = pd.read_csv(file_info_path)
print(df.head(5))

             File Task Category
0  g0pA_taska.txt    a      non
1  g0pA_taskb.txt    b      cut
2  g0pA_taskc.txt    c    light
3  g0pA_taskd.txt    d    heavy
4  g0pA_taske.txt    e      non


In [ ]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class DocSim:
    def __init__(self, model, stopwords=None):
        self.model = model
        self.stopwords = stopwords if stopwords else set()
    
    def preprocess(self, text):
        words = text.lower().split()
        return [w for w in words if w not in self.stopwords and w in self.model]
    
    def vectorize(self, text):
        words = self.preprocess(text)
        if not words:
            return np.zeros(self.model.vector_size)
        word_vectors = [self.model[w] for w in words]
        return np.mean(word_vectors, axis=0)
    
    def calculate_similarity(self, text1, text2):
        vec1 = self.vectorize(text1).reshape(1, -1)
        vec2 = self.vectorize(text2).reshape(1, -1)
        return cosine_similarity(vec1, vec2)[0][0]

# 3. Khởi tạo model và DocSim instance
print("\nLoading Word2Vec model...")
w2v_model = KeyedVectors.load_word2vec_format(
    'GoogleNews-vectors-negative300.bin', 
    binary=True
)

stopwords = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for'])
doc_sim = DocSim(w2v_model, stopwords)

# 4. Phát hiện Plagiarism
def read_file(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def detect_plagiarism(source_file, target_files, threshold=0.7):
    source_text = read_file(source_file)
    results = []
    
    for target_file in target_files:
        target_text = read_file(target_file)
        similarity = doc_sim.calculate_similarity(source_text, target_text)
        results.append({
            'file': os.path.basename(target_file),
            'similarity': similarity,
            'plagiarism': similarity >= threshold
        })
    
    return pd.DataFrame(results).sort_values('similarity', ascending=False)

# Ví dụ sử dụng
source = os.path.join(data_path, "g0pA_taska.txt")
targets = [os.path.join(data_path, f) for f in df['File'][:5]]

print("\nPlagiarism Detection Results:")
results = detect_plagiarism(source, targets, threshold=0.7)
print(results)